# Network Intrusion Detection ML Challenge: Night 1 (EDA + baseline)

**Task:** 4-class classification (Benign / Availability / Reconnaissance / Spoofing), metric = **macro F1**.
**Catch:** the hidden test set contains behavior profiles that are *not* in train, so random-CV scores will look great and mean little.

**Tonight's plan**
1. Load data and run basic sanity checks
2. Look at categoricals and the port columns (likely shortcut features)
3. Train-vs-test shift check (adversarial validation)
4. Baseline LightGBM: stratified CV vs a crude "shifted" CV, to see how optimistic random CV is
5. Write one baseline `submission.csv` (use 1 of your 5 daily submissions on it)

Tomorrow: build a proper grouped CV, test feature sets, add models, blend.

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SEED = 42
TARGET = "Target"
ID = "id"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"./{name}") + glob.glob(f"./data/{name}") + glob.glob(f"../data/{name}") + glob.glob(f"../../data/{name}")
    if not hits:
        raise FileNotFoundError(f"{name} not found. Add the competition data to the notebook.")
    return hits[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
sample = pd.read_csv(find("sample_submission.csv"))

print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)
train.head()

## 1. Basic checks

In [ ]:
feats_all = [c for c in train.columns if c not in (ID, TARGET)]
print("n features:", len(feats_all))

print("\nClass distribution (train):")
print(pd.concat([train[TARGET].value_counts(), train[TARGET].value_counts(normalize=True).round(4)], axis=1,
                keys=["count", "share"]))

num_cols = train[feats_all].select_dtypes("number").columns.tolist()
cat_cols = train[feats_all].select_dtypes(exclude="number").columns.tolist()
print("\nnumeric:", len(num_cols), "| categorical:", cat_cols)

print("\nMissing values (top):")
print(train[feats_all].isna().sum().sort_values(ascending=False).head(8))

print("\nInfinite values (top):")
print(np.isinf(train[num_cols]).sum().sort_values(ascending=False).head(8))

print("\nDuplicate rows (ignoring id):", train[feats_all + [TARGET]].duplicated().sum())
const = [c for c in feats_all if train[c].nunique(dropna=False) <= 1]
print("Constant columns:", const)

In [ ]:
# Do identical feature rows ever carry different labels? (label noise / ambiguity)
g = train.groupby(feats_all, dropna=False)[TARGET].nunique()
print("Feature-identical groups with >1 label:", int((g > 1).sum()), "of", len(g))

## 2. Categoricals and port columns

In [ ]:
for c in cat_cols:
    tr_vals, te_vals = set(train[c].dropna().unique()), set(test[c].dropna().unique())
    print(f"=== {c}: train {len(tr_vals)} values | test {len(te_vals)} values")
    print("  only in test :", sorted(te_vals - tr_vals))
    print("  only in train:", sorted(tr_vals - te_vals))
    print(pd.crosstab(train[c], train[TARGET]).head(15), "\n")

In [ ]:
# Port columns: likely shortcut features
for c in ["id.orig_p", "id.resp_p"]:
    if c not in train.columns:
        continue
    tr_set = set(train[c].unique())
    cover = test[c].isin(tr_set).mean()
    print(f"{c}: {train[c].nunique()} unique in train, {test[c].nunique()} in test | "
          f"{cover:.1%} of test rows have a port value seen in train")

if "id.resp_p" in train.columns:
    print("\nTop response ports per class:")
    for cls, d in train.groupby(TARGET):
        print(cls, d["id.resp_p"].value_counts().head(5).to_dict())

In [ ]:
# Behavior of key features per class (median)
key = [c for c in ["flow_duration", "fwd_pkts_tot", "bwd_pkts_tot", "flow_pkts_per_sec",
                   "payload_bytes_per_second", "flow_SYN_flag_count"] if c in train.columns]
train.groupby(TARGET)[key].median()

In [ ]:
# Skewness: which numeric features are heavy-tailed (candidates for log1p)
skew = train[num_cols].replace([np.inf, -np.inf], np.nan).skew().sort_values(ascending=False)
print(skew.head(12))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, c in zip(axes, key[:3]):
    for cls, d in train.groupby(TARGET):
        ax.hist(np.log1p(d[c].clip(lower=0)), bins=40, alpha=0.5, label=cls)
    ax.set_title(f"log1p({c})")
axes[0].legend()
plt.tight_layout()
plt.show()

## 3. Train vs test shift (adversarial validation)

Train a model to tell train rows from test rows. AUC ~0.5 means no shift; AUC near 1 means the test set is clearly different.
The features it leans on are the ones most likely to hurt generalization.

In [ ]:
def prepare(train, test, drop=()):
    """Same feature matrix for train and test; categoricals share one category set; inf -> NaN."""
    feats = [c for c in train.columns if c not in (ID, TARGET) and c not in drop]
    full = pd.concat([train[feats], test[feats]], ignore_index=True)
    full = full.replace([np.inf, -np.inf], np.nan)
    for c in full.select_dtypes(exclude="number").columns:
        full[c] = full[c].astype("category")
    return full.iloc[:len(train)].reset_index(drop=True), full.iloc[len(train):].reset_index(drop=True)

X_tr, X_te = prepare(train, test)
X_adv = pd.concat([X_tr, X_te], ignore_index=True)
y_adv = np.r_[np.zeros(len(X_tr)), np.ones(len(X_te))]

adv_params = dict(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)

oof = np.zeros(len(y_adv))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_adv, y_adv):
    m = lgb.LGBMClassifier(**adv_params).fit(X_adv.iloc[tr], y_adv[tr])
    oof[va] = m.predict_proba(X_adv.iloc[va])[:, 1]
print("Adversarial AUC (train vs test):", round(roc_auc_score(y_adv, oof), 4))

m = lgb.LGBMClassifier(**adv_params).fit(X_adv, y_adv)
imp = pd.Series(m.booster_.feature_importance("gain"), index=X_adv.columns).sort_values(ascending=False)
print("\nFeatures that best separate train from test:")
print((imp / imp.sum()).head(15).round(4))
adv_top = imp.head(15).index.tolist()

In [ ]:
# Which test rows look least like train? (score from the adversarial model, out-of-fold)
test_adv_score = pd.Series(oof[len(X_tr):], name="p_looks_like_test")
print(test_adv_score.describe().round(3))
print("Share of test rows with p > 0.9:", round((test_adv_score > 0.9).mean(), 3))

## 4. Baseline LightGBM: how optimistic is random CV?

Two CVs side by side:
- **Stratified K-fold:** what most people will trust (and be fooled by)
- **Grouped by `id.resp_p`:** a crude stand-in for "unseen behavior"; whole ports are held out together.

A big gap between the two means the model is leaning on identity-like features rather than behavior.
This crude grouping is only a stand-in; tomorrow you'll build better groups by clustering within each class.

In [ ]:
classes = sorted(train[TARGET].unique())
cls2i = {c: i for i, c in enumerate(classes)}
y = train[TARGET].map(cls2i).values
print(cls2i)

params = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)

def cv_macro_f1(X, y, splitter, groups=None):
    pred = np.zeros(len(y), dtype=int)
    for tr, va in splitter.split(X, y, groups):
        mdl = lgb.LGBMClassifier(**params).fit(X.iloc[tr], y[tr])
        pred[va] = mdl.predict(X.iloc[va])
    return f1_score(y, pred, average="macro"), pred

f1_strat, pred_strat = cv_macro_f1(X_tr, y, StratifiedKFold(5, shuffle=True, random_state=SEED))
print(f"Stratified 5-fold macro-F1: {f1_strat:.4f}")

if "id.resp_p" in train.columns:
    f1_grp, pred_grp = cv_macro_f1(X_tr, y, GroupKFold(5), groups=train["id.resp_p"].values)
    print(f"Grouped-by-resp_p 5-fold macro-F1: {f1_grp:.4f}")

In [ ]:
# Same two CVs without the port columns: does dropping them help under shift?
drop_ports = [c for c in ["id.orig_p", "id.resp_p"] if c in train.columns]
X_tr_np, X_te_np = prepare(train, test, drop=drop_ports)

f1_strat_np, _ = cv_macro_f1(X_tr_np, y, StratifiedKFold(5, shuffle=True, random_state=SEED))
print(f"[no ports] Stratified macro-F1: {f1_strat_np:.4f}")
if "id.resp_p" in train.columns:
    f1_grp_np, pred_grp_np = cv_macro_f1(X_tr_np, y, GroupKFold(5), groups=train["id.resp_p"].values)
    print(f"[no ports] Grouped-by-resp_p macro-F1: {f1_grp_np:.4f}")

In [ ]:
# Per-class view under the grouped CV (where the model struggles under shift)
if "id.resp_p" in train.columns:
    print(classification_report(y, pred_grp, target_names=classes, digits=3))
    cm = pd.DataFrame(confusion_matrix(y, pred_grp), index=classes, columns=classes)
    print(cm)

## 5. Baseline submission

Fit on all training data with all features and write `submission.csv`.
Submit it once and note the public LB score. Compare it with the two CV numbers above:
if the leaderboard is far below stratified CV and close to the grouped one, shift-aware validation is the right direction.

In [ ]:
final = lgb.LGBMClassifier(**params).fit(X_tr, y)
test_pred = final.predict(X_te)
labels = np.array(classes)[test_pred]

sub = pd.DataFrame({ID: test[ID].values, TARGET: labels})
path = os.path.join(OUT, "submission.csv")
sub.to_csv(path, index=False)

# Format checks against the sample submission
assert list(sub.columns) == list(sample.columns) == [ID, TARGET]
assert len(sub) == len(sample) and set(sub[ID]) == set(sample[ID])
assert sub[TARGET].isin(classes).all()
print("Saved", path)
print(sub[TARGET].value_counts(normalize=True).round(3))

## Notes to carry into tomorrow

- Record: stratified CV, grouped CV, no-ports variants, and the public LB score of this baseline.
- Look at the adversarial top features: candidates to drop or transform.
- Check the test predicted-class mix above. A big skew (e.g. almost everything Benign) suggests unseen attacks are being missed.
- Next: cluster within each class to build pseudo behavior groups, run GroupKFold on those, then compare feature sets and models (ExtraTrees, regularized LightGBM, logistic/MLP) and blend.